<a href="https://colab.research.google.com/github/Ilhamlafeer/Airline_Sentiment_Analysis_Using_BERT/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pandas scikit-learn datasets transformers torch accelerate streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 118.8 MB/s eta 0:00:00


In [ ]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [ ]:
import os
import json
import numpy as np
import pandas as pd

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [ ]:
# CONFIGURATION

DATA_PATH = "/content/drive/MyDrive/NLP/Tweets.csv"
MODEL_NAME = "bert-base-uncased"
OUTPUT_DIR = "/content/drive/MyDrive/NLP/airline_sentiment_bert"

MAX_LENGTH = 128
TEST_SIZE = 0.2
RANDOM_STATE = 42

In [ ]:
# LOAD DATA

print("Loading dataset...")

df = pd.read_csv(DATA_PATH)

# Keep only required columns
df = df[["text", "airline_sentiment"]]

# Remove missing values
df = df.dropna()

# Remove duplicate tweets
df = df.drop_duplicates()

print(f"Dataset size: {len(df)}")

print("\nSentiment distribution:")
print(df["airline_sentiment"].value_counts())

Loading dataset...
Dataset size: 14452

Sentiment distribution:
airline_sentiment
negative    9087
neutral     3067
positive    2298
Name: count, dtype: int64


In [ ]:
# ENCODE LABELS
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["airline_sentiment"]
)

print("\nLabel mapping:")

for index, label in enumerate(label_encoder.classes_):
    print(f"{index} -> {label}")


# Save label mapping
os.makedirs(OUTPUT_DIR, exist_ok=True)

label_mapping = {
    str(index): label
    for index, label in enumerate(label_encoder.classes_)
}

with open(
    os.path.join(OUTPUT_DIR, "label_mapping.json"),
    "w"
) as f:
    json.dump(label_mapping, f, indent=4)


Label mapping:
0 -> negative
1 -> neutral
2 -> positive


In [ ]:
# 4. TRAIN / TEST SPLIT
train_df, test_df = train_test_split(
    df[["text", "label"]],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

print("\nTraining samples:", len(train_df))
print("Testing samples:", len(test_df))


# Convert Pandas to Hugging Face Dataset

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)


Training samples: 11561
Testing samples: 2891


In [ ]:
# LOAD BERT TOKENIZER
print("\nLoading tokenizer...")

tokenizer = BertTokenizer.from_pretrained(
    MODEL_NAME
)


Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# TOKENIZATION
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )


print("Tokenizing dataset...")

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)


# Remove text column
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])


# Set PyTorch format
train_dataset.set_format("torch")
test_dataset.set_format("torch")

Tokenizing dataset...


Map:   0%|          | 0/11561 [00:00<?, ? examples/s]

Map:   0%|          | 0/2891 [00:00<?, ? examples/s]

In [ ]:
# 7. LOAD PRETRAINED BERT
print("\nLoading BERT model...")

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_)
)



Loading BERT model...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# METRICS

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

    learning_rate=2e-5,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    logging_steps=50,

    metric_for_best_model="f1",
    greater_is_better=True,

    save_total_limit=2,

    seed=RANDOM_STATE,

    fp16=True,

    report_to="none"
)

In [ ]:
# TRAINER

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)


In [ ]:
# TRAIN

print("\nStarting BERT training...")

trainer.train()



Starting BERT training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.385423,0.479131,0.839156,0.841560,0.839156,0.838797


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.385423,0.479131,0.839156,0.841560,0.839156,0.838797
2,0.301171,0.552340,0.847458,0.848970,0.847458,0.847261
3,0.206031,0.691668,0.849533,0.849013,0.849533,0.849165


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=4338, training_loss=0.33676355585297885, metrics={'train_runtime': 365.0039, 'train_samples_per_second': 95.021, 'train_steps_per_second': 11.885, 'total_flos': 2281390666765056.0, 'train_loss': 0.33676355585297885, 'epoch': 3.0})

In [ ]:
# EVALUATION

print("\nEvaluating model...")

results = trainer.evaluate()

print("\nEvaluation Results:")

for key, value in results.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

    else:
        print(f"{key}: {value}")


Evaluating model...


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.206031,0.691668,3,0.849533,0.849013,0.849533,0.849165



Evaluation Results:
eval_loss: 0.6917
eval_accuracy: 0.8495
eval_precision: 0.8490
eval_recall: 0.8495
eval_f1: 0.8492


In [ ]:
# DETAILED CLASSIFICATION REPORT

print("\nGenerating classification report...")

predictions = trainer.predict(test_dataset)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=1
)

true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=label_encoder.classes_
    )
)



Generating classification report...


              precision    recall  f1-score   support

    negative       0.91      0.91      0.91      1818
     neutral       0.72      0.69      0.71       613
    positive       0.78      0.81      0.79       460

    accuracy                           0.85      2891
   macro avg       0.80      0.81      0.80      2891
weighted avg       0.85      0.85      0.85      2891



In [ ]:
# CONFUSION MATRIX
cm = confusion_matrix(
    true_labels,
    predicted_labels
)

print("\nConfusion Matrix:")
print(cm)


Confusion Matrix:
[[1657  114   47]
 [ 127  426   60]
 [  36   51  373]]


In [ ]:
# SAVE MODEL
print("\nSaving model...")

model.save_pretrained(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)

print("\nModel saved to:")
print(OUTPUT_DIR)

print("\nTraining completed successfully!")


Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
/content/drive/MyDrive/NLP/airline_sentiment_bert

Training completed successfully!
